---
# **Ambient Config -**
---

In [ ]:
!pip install tensorflow pymongo pandas numpy scikit-learn matplotlib seaborn fastapi uvicorn pyngrok nest-asyncio --quiet

import time
import tensorflow as tf
import keras
import pymongo
import pandas as pd
import numpy as np
import urllib.request, zipfile, os
import ssl
import random
import pickle
import uvicorn, asyncio, json
import nest_asyncio
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pyngrok import ngrok
from datetime import datetime, timezone
from google.colab import userdata
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure
from keras import layers
from collections import defaultdict

print(f"TensorFlow: {tf.__version__}")
print(f"PyMongo:    {pymongo.__version__}")
print(f"Available GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow: 2.20.0
PyMongo:    4.17.0
Available GPU: False


---
# **RecSys -**
---

In [ ]:
URL  = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
DEST = "/content/ml-1m.zip"

if not os.path.exists("/content/ml-1m"):
    print("Downloading MovieLens 1M")
    urllib.request.urlretrieve(URL, DEST)
    with zipfile.ZipFile(DEST) as z:
        z.extractall("/content")

SEP = "::"

ratings = pd.read_csv("/content/ml-1m/ratings.dat",
                      sep=SEP, engine="python",
                      names=["user_id","movie_id","rating","timestamp"])

movies  = pd.read_csv("/content/ml-1m/movies.dat",
                      sep=SEP, engine="python", encoding="latin-1",
                      names=["movie_id","title","genres"])

users   = pd.read_csv("/content/ml-1m/users.dat",
                      sep=SEP, engine="python",
                      names=["user_id","gender","age","occupation","zip"])

print(f"Ratings: {ratings.shape}")
print(f"Movies:  {movies.shape}")
print(f"Users:   {users.shape}")
ratings.head()

Ratings: (1000209, 4)
Movies:  (3883, 3)
Users:   (6040, 5)


,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [ ]:
MONGO_URI = userdata.get("MONGO_URI")

try:
    client = MongoClient(
        MONGO_URI,
        serverSelectionTimeoutMS=10000,
        tls=True,
        tlsAllowInvalidCertificates=True
    )
    client.admin.command("ping")
    print("Connecting to MongoDB Atlas.")
except ConnectionFailure as e:
    print(f"Connection error: {e}")

db = client["recsys"]
col_users = db["users"]
col_movies = db["movies"]
col_ratings = db["ratings"]
col_sessions = db["sessions"]

Connecting to MongoDB Atlas.


In [ ]:
# DO NOT RUN IT MORE THEN ONCE, UNNECESSARY AND CONSUMES A LOT OF TIME. I recommend that you comment on the entire session after running it for the first time.

def bulk_insert(collection, docs, batch_size=5000, label=""):
    total = len(docs)
    inseridos = 0
    t0 = time.time()

    for i in range(0, total, batch_size):
        lote = docs[i : i + batch_size]
        collection.insert_many(lote, ordered=False)
        inseridos += len(lote)
        pct = inseridos / total * 100
        elapsed = time.time() - t0
        restante = (elapsed / inseridos) * (total - inseridos) if inseridos > 0 else 0
        print(f"\r  {label}: {inseridos}/{total} ({pct:.0f}%) — {restante:.0f}s remaining", end="")

    print(f"\r  {label}: {inseridos} docs in {time.time()-t0:.1f}s ")

col_users.drop()
col_movies.drop()
col_ratings.drop()

col_users = db["users"]
col_movies = db["movies"]
col_ratings = db["ratings"]
col_sessions = db["sessions"]

movies_docs = movies.copy()
movies_docs["genres"] = movies_docs["genres"].str.split("|")

bulk_insert(col_users,   users.to_dict("records"), batch_size=5000, label="Users")
bulk_insert(col_movies,  movies_docs.to_dict("records"), batch_size=5000, label="Movies")
bulk_insert(col_ratings, ratings.to_dict("records"), batch_size=5000, label="Ratings")

col_ratings.create_index([("user_id", 1), ("timestamp", -1)])
col_movies.create_index([("movie_id", 1)])

print(f"\nRatings on Atlas: {col_ratings.count_documents({})}")
print(f"Users:    {col_users.count_documents({})}")
print(f"Movies:   {col_movies.count_documents({})}")
print(f"Ratings:  {col_ratings.count_documents({})}")

print(f"\nIndexes in classifications: {list(col_ratings.index_information().keys())}")

  Users: 6040 docs in 9.5s 
  Movies: 3883 docs in 6.2s 
  Ratings: 1000209 docs in 1138.3s 

Ratings on Atlas: 1000209
Users:    6040
Movies:   3883
Ratings:  1000209

Indexes in classifications: ['_id_', 'user_id_1_timestamp_-1']


In [ ]:
# DO NOT RUN IT MORE THEN ONCE, UNNECESSARY AND CONSUMES A LOT OF TIME ALSO. Same as before, comment the entire session after running it once.

def build_sessions(batch_size=500):
    movies_lookup = {
        doc["movie_id"]: {"title": doc["title"], "genres": doc["genres"]}
        for doc in col_movies.find({}, {"movie_id":1, "title":1, "genres":1, "_id":0})
    }

    all_users = list(col_users.find({}, {"_id":0}))
    total = len(all_users)

    sessions_batch = []
    t0 = time.time()

    for idx, user in enumerate(all_users):
        uid = user["user_id"]

        raw = list(
            col_ratings.find(
                {"user_id": uid},
                {"movie_id":1, "rating":1, "timestamp":1, "_id":0}
            ).sort("timestamp", -1).limit(50)
        )

        interactions = []
        for r in raw:
            mid = r["movie_id"]
            movie_info = movies_lookup.get(mid, {"title": "Unknown", "genres": []})
            interactions.append({
                "movie_id": mid,
                "title": movie_info["title"],
                "genres": movie_info["genres"],
                "rating": r["rating"],
                "timestamp": r["timestamp"]
            })

        session_doc = {
            "user_id": uid,
            "gender": user["gender"],
            "age": user["age"],
            "occupation": user["occupation"],
            "last_updated": datetime.now(timezone.utc),
            "interactions": interactions
        }
        sessions_batch.append(session_doc)

        if len(sessions_batch) == batch_size:
            col_sessions.insert_many(sessions_batch, ordered=False)
            sessions_batch = []

        if (idx + 1) % 1000 == 0:
            pct = (idx + 1) / total * 100
            print(f"  {idx+1}/{total} ({pct:.0f}%)")

    if sessions_batch:
        col_sessions.insert_many(sessions_batch, ordered=False)

    print(f"\nSessions Created: {col_sessions.count_documents({})} em {time.time()-t0:.1f}s")

col_sessions.drop()
col_sessions = db["sessions"]
build_sessions()

  1000/6040 (17%)
  2000/6040 (33%)
  3000/6040 (50%)
  4000/6040 (66%)
  5000/6040 (83%)
  6000/6040 (99%)

Sessions Created: 6040 em 908.9s


In [ ]:
all_ratings = list(
    col_ratings.find({}, {"user_id":1, "movie_id":1, "rating":1, "timestamp":1, "_id":0})
)
df = pd.DataFrame(all_ratings)
print(f"Shape: {df.shape}")
print(df.head(3))

unique_users  = sorted(df["user_id"].unique())
unique_movies = sorted(df["movie_id"].unique())

user2idx  = {uid: idx for idx, uid in enumerate(unique_users)}
movie2idx = {mid: idx for idx, mid in enumerate(unique_movies)}
idx2movie = {idx: mid for mid, idx in movie2idx.items()}

NUM_USERS  = len(user2idx)
NUM_MOVIES = len(movie2idx)

df["user_idx"]  = df["user_id"].map(user2idx)
df["movie_idx"] = df["movie_id"].map(movie2idx)

Shape: (1000209, 4)
   user_id  movie_id  rating  timestamp
0        1      1193       5  978300760
1        1       661       3  978302109
2        1       914       3  978301968


In [ ]:
random.seed(42)
np.random.seed(42)

user_watched = df.groupby("user_idx")["movie_idx"].apply(set).to_dict()
df_pos = df[df["rating"] >= 4].copy()

movie_counts = df["movie_idx"].value_counts()
movie_pop_idxs = movie_counts.index.tolist()
movie_pop_weights = movie_counts.values / movie_counts.values.sum()

def sample_negative_hard(user_idx):
    watched = user_watched.get(user_idx, set())
    for _ in range(50):
        candidate = np.random.choice(movie_pop_idxs, p=movie_pop_weights)
        if candidate not in watched:
            return candidate
    return random.choice([m for m in range(NUM_MOVIES) if m not in watched])

t0 = time.time()
train_users = df_pos["user_idx"].values
train_pos = df_pos["movie_idx"].values
train_neg = np.array([sample_negative_hard(u) for u in train_users])

In [ ]:
BATCH_SIZE = 1024
VAL_SPLIT = 0.1

n_total = len(train_users)
n_val = int(n_total * VAL_SPLIT)
n_train = n_total - n_val

perm = np.random.permutation(n_total)
train_users = train_users[perm]
train_pos = train_pos[perm]
train_neg = train_neg[perm]

users_train, users_val = train_users[:n_train], train_users[n_train:]
pos_train, pos_val = train_pos[:n_train], train_pos[n_train:]
neg_train, neg_val = train_neg[:n_train], train_neg[n_train:]

def make_dataset(users, positives, negatives, shuffle=True):

    ds = tf.data.Dataset.from_tensor_slices({
        "user_idx": users.astype(np.int32),
        "pos_movie_idx": positives.astype(np.int32),
        "neg_movie_idx": negatives.astype(np.int32),
    })
    if shuffle:
        ds = ds.shuffle(buffer_size=50_000, seed=42)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

ds_train = make_dataset(users_train, pos_train, neg_train, shuffle=True)
ds_val = make_dataset(users_val,   pos_val,   neg_val,   shuffle=False)

In [ ]:
EMBEDDING_DIM = 64
TOWER_DIM     = 128

def build_user_tower(num_users, emb_dim, tower_dim):
    inp = keras.Input(shape=(), dtype=tf.int32, name="user_input")
    x = layers.Embedding(num_users, emb_dim, name="user_embedding")(inp)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(tower_dim)(x)
    x = layers.Lambda(lambda v: tf.math.l2_normalize(v, axis=-1), name="user_vec")(x)
    return keras.Model(inp, x, name="user_tower")

def build_item_tower(num_items, emb_dim, tower_dim):
    inp = keras.Input(shape=(), dtype=tf.int32, name="item_input")
    x = layers.Embedding(num_items, emb_dim, name="item_embedding")(inp)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(tower_dim)(x)
    x = layers.Lambda(lambda v: tf.math.l2_normalize(v, axis=-1), name="item_vec")(x)
    return keras.Model(inp, x, name="item_tower")

user_tower = build_user_tower(NUM_USERS, EMBEDDING_DIM, TOWER_DIM)
item_tower = build_item_tower(NUM_MOVIES, EMBEDDING_DIM, TOWER_DIM)

print(user_tower.summary())
print(item_tower.summary())

Model: "user_tower"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ user_input (InputLayer)         │ (None)                 │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ user_embedding (Embedding)      │ (None, 64)             │       386,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ user_vec (Lambda)               │ (None, 128)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 411,392 (1.57 MB)

 Trainable params: 411,392 (1.57 MB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ item_input (InputLayer)         │ (None)                 │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ item_embedding (Embedding)      │ (None, 64)             │       237,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ item_vec (Lambda)               │ (None, 128)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 262,016 (1023.50 KB)

 Trainable params: 262,016 (1023.50 KB)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
class TwoTowerModel(keras.Model):
    def __init__(self, user_tower, item_tower, **kwargs):
        super().__init__(**kwargs)
        self.user_tower = user_tower
        self.item_tower = item_tower
        self.loss_tracker = keras.metrics.Mean(name="loss")

    @property
    def metrics(self):
        return [self.loss_tracker]

    def compute_loss_from_inputs(self, inputs, training=False):
        user_vec = self.user_tower(inputs["user_idx"], training=training)
        pos_vec = self.item_tower(inputs["pos_movie_idx"], training=training)
        neg_vec = self.item_tower(inputs["neg_movie_idx"], training=training)

        pos_score = tf.reduce_sum(user_vec * pos_vec, axis=1)
        neg_score = tf.reduce_sum(user_vec * neg_vec, axis=1)

        return -tf.reduce_mean(tf.math.log(tf.sigmoid(pos_score - neg_score) + 1e-8))

    def train_step(self, data):
        with tf.GradientTape() as tape:
            loss = self.compute_loss_from_inputs(data, training = True)

        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    def test_step(self, data):
        loss = self.compute_loss_from_inputs(data, training=False)
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    def call(self, inputs, training=False):
        user_vec = self.user_tower(inputs["user_idx"], training=training)
        pos_vec = self.item_tower(inputs["pos_movie_idx"], training=training)
        neg_vec = self.item_tower(inputs["neg_movie_idx"], training=training)
        pos_score = tf.reduce_sum(user_vec * pos_vec, axis=1, keepdims=True)
        neg_score = tf.reduce_sum(user_vec * neg_vec, axis=1, keepdims=True)
        return tf.concat([pos_score, neg_score], axis=1)

user_tower = build_user_tower(NUM_USERS, EMBEDDING_DIM, TOWER_DIM)
item_tower = build_item_tower(NUM_MOVIES, EMBEDDING_DIM, TOWER_DIM)

model = TwoTowerModel(user_tower, item_tower, name="two_tower")
model.compile(optimizer = keras.optimizers.Adam(learning_rate=1e-3))

for batch in ds_train.take(1):
    _ = model(batch, training=False)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

print("Initializing training...")
history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)

Initializing training...
Epoch 1/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 20s 34ms/step - loss: 0.6102 - val_loss: 0.5611 - learning_rate: 0.0010
Epoch 2/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 17s 34ms/step - loss: 0.5489 - val_loss: 0.5432 - learning_rate: 0.0010
Epoch 3/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 19s 37ms/step - loss: 0.5332 - val_loss: 0.5333 - learning_rate: 0.0010
Epoch 4/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 17s 34ms/step - loss: 0.5223 - val_loss: 0.5267 - learning_rate: 0.0010
Epoch 5/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 17s 34ms/step - loss: 0.5161 - val_loss: 0.5241 - learning_rate: 0.0010
Epoch 6/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - loss: 0.5126 - val_loss: 0.5224 - learning_rate: 0.0010
Epoch 7/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 22s 37ms/step - loss: 0.5092 - val_loss: 0.5174 - learning_rate: 0.0010
Epoch 8/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 19s 33ms/step - loss: 0.5044 - val_loss: 0.5126 - learning_rate: 0.0010
Epoch 9/30
506/506 ━━━━━━━━━━━━━━━━━━━━ 18s 35ms/step - loss: 0.5011 - val_loss

In [ ]:
SAVE_DIR = "/content/two_tower_model"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_weights(f"{SAVE_DIR}/model_weights.weights.h5")

item_embeddings = item_tower.predict(
    np.arange(NUM_MOVIES, dtype=np.int32),
    batch_size=512,
    verbose=0
)
user_embeddings = user_tower.predict(
    np.arange(NUM_USERS, dtype=np.int32),
    batch_size=512,
    verbose=0
)

np.save(f"{SAVE_DIR}/item_embeddings.npy", item_embeddings)
np.save(f"{SAVE_DIR}/user_embeddings.npy", user_embeddings)

with open(f"{SAVE_DIR}/user2idx.pkl", "wb") as f:
    pickle.dump(user2idx, f)
with open(f"{SAVE_DIR}/movie2idx.pkl", "wb") as f:
    pickle.dump(movie2idx, f)
with open(f"{SAVE_DIR}/idx2movie.pkl", "wb") as f:
    pickle.dump(idx2movie, f)

In [ ]:
def recomendar(user_id, top_k=10):
    if user_id not in user2idx:
        print(f"User {user_id} not found.")
        return

    u_idx = user2idx[user_id]

    u_vec = user_embeddings[u_idx]

    scores = item_embeddings @ u_vec

    watched_idxs = list(user_watched[u_idx])
    scores[watched_idxs] = -np.inf

    top_idxs = np.argsort(scores)[::-1][:top_k]

    movies_lookup = {
        doc["movie_id"]: doc["title"]
        for doc in col_movies.find({}, {"movie_id":1, "title":1, "_id":0})
    }

    print(f"\nRecommendations for user {user_id}:")
    print(f"{'Rank':<5} {'Score':>6} Movie Title")
    print("-" * 50)
    for rank, idx in enumerate(top_idxs, 1):
        mid   = idx2movie[idx]
        title = movies_lookup.get(mid, "Unknown")
        print(f"  {rank:<3}  {scores[idx]:>6.3f}  {title}")

    session = col_sessions.find_one({"user_id": user_id})
    print(f"\nLast 5 movies watched that have a rating ≥ 4:")
    count = 0
    for i in session["interactions"]:
        if i["rating"] >= 4:
            print(f"  [{i['rating']}★] {i['title']}")
            count += 1
        if count == 5:
            break

recomendar(user_id=1)
recomendar(user_id=200)


Recommendations for user 1:
Rank   Score Movie Title
--------------------------------------------------
  1     0.971  Field of Dreams (1989)
  2     0.959  Dave (1993)
  3     0.953  Pee-wee's Big Adventure (1985)
  4     0.948  Madame Butterfly (1995)
  5     0.935  Sixteen Candles (1984)
  6     0.932  Much Ado About Nothing (1993)
  7     0.930  Parenthood (1989)
  8     0.922  Four Weddings and a Funeral (1994)
  9     0.921  American President, The (1995)
  10    0.905  Rob Roy (1995)

Last 5 movies watched that have a rating ≥ 4:
  [5★] Pocahontas (1995)
  [4★] Hercules (1997)
  [4★] Mulan (1998)
  [5★] Bug's Life, A (1998)
  [4★] Antz (1998)

Recommendations for user 200:
Rank   Score Movie Title
--------------------------------------------------
  1     0.997  Seventh Sign, The (1988)
  2     0.996  Varsity Blues (1999)
  3     0.995  Empire Records (1995)
  4     0.989  For Love of the Game (1999)
  5     0.986  Spiders, The (Die Spinnen, 1. Teil: Der Goldene See) (1919)
  6

In [ ]:
def evaluate_model_protocol(n_users_eval=500, K=10, n_candidates=100):

    recall_list = []
    ndcg_list = []

    eligible = [
        uid for uid, watched in user_watched.items()
        if len(watched) >= 5
    ]
    sampled_users = np.random.choice(
        eligible,
        size=min(n_users_eval, len(eligible)),
        replace=False
    )

    for u_idx in sampled_users:
        u_idx = int(u_idx)
        uid_original = int(unique_users[u_idx])

        interactions = list(
            col_ratings.find(
                {"user_id": uid_original, "rating": {"$gte": 4}},
                {"movie_id": 1, "timestamp": 1, "_id": 0}
            ).sort("timestamp", -1)
        )

        if len(interactions) < 2:
            continue

        gt_movie_id  = interactions[0]["movie_id"]
        gt_movie_idx = movie2idx.get(gt_movie_id)
        if gt_movie_idx is None:
            continue

        watched = user_watched[u_idx]
        negatives = []
        while len(negatives) < (n_candidates - 1):
            candidate = random.randint(0, NUM_MOVIES - 1)
            if candidate not in watched and candidate != gt_movie_idx:
                negatives.append(candidate)

        candidates = np.array([gt_movie_idx] + negatives)

        u_vec  = user_embeddings[u_idx]
        scores = item_embeddings[candidates] @ u_vec

        ranked = np.argsort(scores)[::-1]
        rank_of_positive = np.where(ranked == 0)[0][0] + 1

        hit = int(rank_of_positive <= K)
        recall_list.append(hit)

        ndcg = 1.0 / np.log2(rank_of_positive + 1) if hit else 0.0
        ndcg_list.append(ndcg)

    recall = np.mean(recall_list)
    ndcg = np.mean(ndcg_list)

    print(f"Rated Users: {len(recall_list)}")
    print(f"Recall@{K}: {recall:.4f} ({recall*100:.1f}%)")
    print(f"NDCG@{K}: {ndcg:.4f}")
    return recall, ndcg

recall, ndcg = evaluate_model_protocol(n_users_eval=500, K=10)

Rated Users: 498
Recall@10: 0.5040 (50.4%)
NDCG@10: 0.2683


In [ ]:
ks  = [1, 5, 10, 20, 50]
recalls = []
ndcgs   = []

print("Calculating metrics:")
for k in ks:
    r, n = evaluate_model_protocol(n_users_eval=500, K=k)
    recalls.append(r)
    ndcgs.append(n)
    print()

Calculating metrics:
Rated Users: 500
Recall@1: 0.0900 (9.0%)
NDCG@1: 0.0900

Rated Users: 500
Recall@5: 0.3220 (32.2%)
NDCG@5: 0.2076

Rated Users: 500
Recall@10: 0.4740 (47.4%)
NDCG@10: 0.2480

Rated Users: 500
Recall@20: 0.6240 (62.4%)
NDCG@20: 0.3081

Rated Users: 499
Recall@50: 0.8978 (89.8%)
NDCG@50: 0.3502



---
#                         **FastAPI -**
---





In [ ]:
nest_asyncio.apply()

SAVE_DIR = "/content/two_tower_model"

arquivos = os.listdir(SAVE_DIR)
print("Available artifacts:")
for f in arquivos:
    size = os.path.getsize(f"{SAVE_DIR}/{f}") / 1024
    print(f"  {f:<35} {size:.1f} KB")

movies_lookup = {
    int(doc["movie_id"]): {
        "title":  doc["title"],
        "genres": doc["genres"]
    }
    for doc in col_movies.find({}, {"movie_id":1, "title":1, "genres":1, "_id":0})
}

with open(f"{SAVE_DIR}/movies_lookup.pkl", "wb") as f:
    pickle.dump(movies_lookup, f)

with open(f"{SAVE_DIR}/unique_users.pkl", "wb") as f:
    pickle.dump(unique_users, f)

print(f"\n{len(movies_lookup)} Saved movie lookups")

Available artifacts:
  idx2movie.pkl                       79.5 KB
  user2idx.pkl                        129.6 KB
  model_weights.weights.h5            7940.6 KB
  unique_users.pkl                    112.2 KB
  movies_lookup.pkl                   216.7 KB
  item_embeddings.npy                 1853.1 KB
  user_embeddings.npy                 3020.1 KB
  movie2idx.pkl                       79.5 KB

3883 Saved movie lookups


In [ ]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

def jsonify(data):
    return JSONResponse(content=json.loads(json.dumps(data, cls=NumpyEncoder)))

SAVE_DIR = "/content/two_tower_model"

item_embeddings = np.load(f"{SAVE_DIR}/item_embeddings.npy")
user_embeddings = np.load(f"{SAVE_DIR}/user_embeddings.npy")

with open(f"{SAVE_DIR}/user2idx.pkl", "rb") as f: user2idx = pickle.load(f)
with open(f"{SAVE_DIR}/idx2movie.pkl", "rb") as f: idx2movie = pickle.load(f)
with open(f"{SAVE_DIR}/movie2idx.pkl", "rb") as f: movie2idx = pickle.load(f)
with open(f"{SAVE_DIR}/movies_lookup.pkl", "rb") as f: movies_lookup = pickle.load(f)
with open(f"{SAVE_DIR}/unique_users.pkl", "rb") as f: unique_users = pickle.load(f)

MONGO_URI = userdata.get("MONGO_URI")
_client = MongoClient(MONGO_URI, tls=True, tlsAllowInvalidCertificates=True)
_db = _client["recsys"]
_ratings = _db["ratings"]
_sessions = _db["sessions"]

print("Loaded artifacts.")
print(f" item_embeddings: {item_embeddings.shape}")
print(f" user_embeddings: {user_embeddings.shape}")

app = FastAPI(title="Two-Tower RecSys API", version="1.0")

@app.get("/")
def root():
    return {"status": "online", "model": "Two-Tower MovieLens 1M"}

@app.get("/recommend/{user_id}")
def recommend(user_id: int, top_k: int = 10):
    if user_id not in user2idx:
        raise HTTPException(status_code=404, detail=f"user_id {user_id} not found.")

    u_idx  = int(user2idx[user_id])
    u_vec  = user_embeddings[u_idx]
    scores = item_embeddings @ u_vec

    watched_raw  = _ratings.find({"user_id": user_id}, {"movie_id": 1, "_id": 0})
    watched_idxs = [movie2idx[d["movie_id"]] for d in watched_raw if d["movie_id"] in movie2idx]
    scores[watched_idxs] = -np.inf

    top_idxs = np.argsort(scores)[::-1][:top_k]

    recommendations = []
    for rank, idx in enumerate(top_idxs, 1):
        mid        = int(idx2movie[int(idx)])
        movie_info = movies_lookup.get(mid, {"title": "Unknown", "genres": []})
        recommendations.append({
            "rank": rank,
            "score": round(float(scores[idx]), 4),
            "movie_id": mid,
            "title": movie_info["title"],
            "genres": movie_info["genres"]
        })

    session = _sessions.find_one({"user_id": user_id}, {"_id": 0})
    recent  = []
    if session:
        for i in session["interactions"][:5]:
            recent.append({
                "title": i["title"],
                "rating": int(i["rating"]),
                "genres": i["genres"]
            })

    return jsonify({
        "user_id": user_id,
        "top_k": top_k,
        "recommendations": recommendations,
        "recent_history": recent
    })

@app.get("/movie/{movie_id}")
def movie_info(movie_id: int):
    info = movies_lookup.get(movie_id)
    if not info:
        raise HTTPException(status_code=404, detail=f"movie_id {movie_id} not found.")
    return jsonify({"movie_id": movie_id, **info})

@app.get("/similar/{movie_id}")
def similar_movies(movie_id: int, top_k: int = 10):
    if movie_id not in movie2idx:
        raise HTTPException(status_code=404, detail=f"movie_id {movie_id} not found.")

    m_idx  = int(movie2idx[movie_id])
    m_vec  = item_embeddings[m_idx]
    scores = item_embeddings @ m_vec
    scores[m_idx] = -np.inf

    top_idxs = np.argsort(scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_idxs, 1):
        mid  = int(idx2movie[int(idx)])
        info = movies_lookup.get(mid, {"title": "Unknown", "genres": []})
        results.append({
            "rank":     rank,
            "score":    round(float(scores[idx]), 4),
            "movie_id": mid,
            "title":    info["title"],
            "genres":   info["genres"]
        })

    source = movies_lookup.get(movie_id, {})
    return jsonify({"source_movie": {"movie_id": movie_id, **source}, "similar": results})

# START FASTAPI SERVER
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
public_url = ngrok.connect(8000)
url = public_url.public_url if hasattr(public_url, "public_url") else str(public_url).split('"')[1]

print(f"\nPublic API:  {url}/docs")
print(f"  GET {url}/recommend/1")
print(f"  GET {url}/similar/1")
print(f"  GET {url}/movie/1")

async def start():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()

asyncio.get_event_loop().run_until_complete(start())

Loaded artifacts.
 item_embeddings: (3706, 128)
 user_embeddings: (6040, 128)

Public API:  https://prize-oxymoron-pusher.ngrok-free.dev/docs
  GET https://prize-oxymoron-pusher.ngrok-free.dev/recommend/1
  GET https://prize-oxymoron-pusher.ngrok-free.dev/similar/1
  GET https://prize-oxymoron-pusher.ngrok-free.dev/movie/1


INFO:     Started server process [11935]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [11935]


KeyboardInterrupt: 